In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("start_month", "2025-01")
dbutils.widgets.text("end_month", "")     # leave blank for only 1 month (such as newest month)
dbutils.widgets.text("sources", "yellow,green")

catalog = dbutils.widgets.get("catalog")

catalog = dbutils.widgets.get("catalog").strip()
start_month = dbutils.widgets.get("start_month").strip()
end_month = dbutils.widgets.get("end_month").strip() or start_month
sources = [s.strip() for s in dbutils.widgets.get("sources").split(",") if s.strip()]

In [0]:
import os, sys, yaml

src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from nyctaxi.landing import month_range
from nyctaxi.silver import *
from pyspark.sql import functions as F

with open("../config/sources.yml") as f:
    config = yaml.safe_load(f)

months = list(month_range(start_month, end_month))

unknown = [s for s in sources if s not in COLUMN_MAPPINGS]
if unknown:
    raise ValueError(f"Unknown trip sources: {unknown}.")

TRIPS = f"{catalog}.silver.trips"
QUARANTINE = f"{catalog}.silver.trips_quarantine"
ZONES = f"{catalog}.silver.taxi_zones"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver")

In [0]:
# for reset if needed
# spark.sql(f"DROP TABLE IF EXISTS {TRIPS}")
# spark.sql(f"DROP TABLE IF EXISTS {QUARANTINE}")

In [0]:
zones = standardize_zone(spark.table(f"{catalog}.bronze.{config['sources']['zones']['bronze_table']}"))

(zones.write
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(ZONES))

print(f"{ZONES}: {spark.table(ZONES).count()} rows")

In [0]:
for month in months:
    for taxi_type in sources:
        bronze = spark.table(f"{catalog}.bronze.{config['sources'][taxi_type]['bronze_table']}")
        tagged = build_silver_trips(bronze, taxi_type, month)
        valid, rejected = split_valid_rejected(tagged)
        merge_trips(spark, valid, TRIPS, taxi_type, month)
        write_quarantine(spark, rejected, QUARANTINE, taxi_type, month)
        print(f"{taxi_type} {month}: done")

In [0]:
print("valid silver rows count by taxi type")
spark.table(TRIPS).groupBy("taxi_type", "source_file").count().orderBy("source_file").show(truncate=False)

print("quarantine rows count by reason")
(spark.table(QUARANTINE).groupBy("taxi_type", "file_month", "dq_reason").count()
     .orderBy("taxi_type", "file_month", F.desc("count")).show(50, truncate=False))

print("write history")
spark.sql(f"DESCRIBE HISTORY {TRIPS}").select("version", "operation", "operationMetrics").show(truncate=False)